# CymbalGoal AlloyDB Workshop—Load Test: Scout Reports into AlloyDB

**Track A, pipeline step 02.** Measures how long it takes to get the staged scouting-report
corpus (38,445 reports, 3,072-dimension embeddings, ~490 MB compressed) into a freshly
provisioned AlloyDB cluster and searchable, so we can decide how a lab should load it.

Cluster creation is **not** timed here. Terraform builds the cluster first; this notebook
starts once it exists.

| What it times | How |
| :-- | :-- |
| **Base tables** | the 8 CymbalGoal tables, client-side `COPY` (the earlier CymbalGoal lab's method) |
| **A · COPY, one file** | the whole corpus streamed from Cloud Storage through this runtime into `COPY` |
| **B · COPY, parallel** | the 14 per-season files, several `COPY` streams at once |
| **C · Import, course bucket** | `gcloud alloydb clusters import` straight from `class-demo` (server-side) |
| **D · Import, project bucket** | copy to the in-project staging bucket Terraform made, then import |
| **Indexes** | B-tree indexes, then **ScaNN** at 1 season, 5 seasons and all seasons |
| **Search check** | a real semantic query through `ai.embedding()`, ScaNN vs exact recall |

| | |
| :-- | :-- |
| **Reads** | `gs://class-demo/alloydb-labs/cymbalgoal/` and the staged corpus under `…/cymbalgoal-alloydb-workshop/scout-reports/staged/` |
| **Writes** | the results file under `…/cymbalgoal-alloydb-workshop/load-tests/`, and copies into this project's own staging bucket. Nothing else. |
| **Database** | its own `cymbalgoal_loadtest` database, so it never touches a lab database |

**Run all.** Each method loads its own table, dropped and recreated each run so every time is a
fresh load. The last cell, **13 · Handback**, holds everything Claude needs with a **Copy** button.

---
## 1 · Configuration

Every method is behind a flag. Turn one off to skip it on a rerun.

In [ ]:
RUN_BASE           = True    # load the 8 CymbalGoal tables (also gives scout_reports its foreign keys)
FRESH_BASE         = False   # True = drop and reload the base tables even if they are already there
RUN_COPY_ONE       = True    # A: COPY the whole corpus, one stream
RUN_COPY_PARALLEL  = True    # B: COPY the per-season files, several streams at once
PARALLEL_WORKERS   = 8
RUN_IMPORT_DIRECT  = True    # C: AlloyDB import straight from the course bucket
RUN_IMPORT_PROJECT = True    # D: copy to the project's staging bucket, then AlloyDB import
RUN_INDEXES        = True    # B-tree + ScaNN builds at three sizes
RUN_SEARCH         = True    # semantic search through ai.embedding(), ScaNN vs exact

In [ ]:
import os, sys, subprocess

if not os.environ.get("CG_FIXTURE"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "google-cloud-alloydb-connector[pg8000]", "pandas"], check=True)
print("client libraries ready")

In [ ]:
# ---- Where things live ---------------------------------------------------------------------
GCS_READ       = "gs://class-demo/alloydb-labs/cymbalgoal"                       # read-only
GCS_WORK       = "gs://class-demo/alloydb-labs/cymbalgoal-alloydb-workshop"
STAGED_PREFIX  = f"{GCS_WORK}/scout-reports/staged"
RESULTS_PREFIX = f"{GCS_WORK}/load-tests"                                       # the only course-bucket write
DB_NAME        = "cymbalgoal_loadtest"

# ---- The event's time budget ---------------------------------------------------------------
# A student clicks Start Lab and waits ~20 minutes. Cluster creation takes ~9 of them, so a load
# that runs as part of that wait has roughly 11 minutes.
BUDGET_MIN = 11.0

# ---- Index and search settings -------------------------------------------------------------
MAINTENANCE_WORK_MEM = "2GB"
EMBED_MODEL = "gemini-embedding-001"
SCANN_SIZES = [("1 season", 1), ("5 seasons", 5), ("all seasons", None)]
SEARCH_QUERIES = [
    "an attacking full-back who creates chances and scores late goals",
    "an experienced centre-back who is dominant in the air",
    "a young creative midfielder worth monitoring for the future",
]

---
## 2 · Environment and guard

Discovers the project, cluster and instance rather than assuming them, and arms a guard so
every Cloud Storage call is checked: reads only from the two course prefixes and this project's
staging bucket, writes only to the results prefix and the staging bucket, and no deletes.

In [ ]:
import csv, gzip, io, json, math, re, threading, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

_FIXTURE = os.environ.get("CG_FIXTURE")          # offline tests only; unset in Colab


def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


BASE = Path("/content/cymbalgoal-loadtest")
try:
    BASE.mkdir(parents=True, exist_ok=True)
except OSError:
    BASE = Path.home() / "cymbalgoal-loadtest"
    BASE.mkdir(parents=True, exist_ok=True)

if _FIXTURE:
    PROJECT, USER, REGION = "fixture-project", "postgres", "us-central1"
    CLUSTER, INSTANCE, PROJECT_NUMBER = "cymbalgoal-cluster", "cymbalgoal-primary", "123"
    SHAPE = {"cpu": 8, "availability": "ZONAL", "version": "POSTGRES_18"}
else:
    PROJECT = sh("gcloud config get-value project")
    USER = sh("gcloud config get-value account")
    PROJECT_NUMBER = sh(f"gcloud projects describe {PROJECT} --format='value(projectNumber)'")
    clusters = json.loads(sh("gcloud alloydb clusters list --format=json") or "[]")
    assert clusters, "No AlloyDB cluster in this project. Apply the Terraform first."
    _cl = clusters[0]
    CLUSTER = _cl["name"].split("/")[-1]
    REGION = _cl["name"].split("/locations/")[1].split("/")[0]
    _inst = [i for i in json.loads(sh(f"gcloud alloydb instances list --cluster={CLUSTER} "
                                      f"--region={REGION} --format=json") or "[]")
             if i.get("instanceType") == "PRIMARY"]
    assert _inst, f"No PRIMARY instance in cluster {CLUSTER}."
    INSTANCE = _inst[0]["name"].split("/")[-1]
    SHAPE = {"cpu": _inst[0].get("machineConfig", {}).get("cpuCount"),
             "availability": _inst[0].get("availabilityType"),
             "version": _cl.get("databaseVersion")}

INSTANCE_URI = f"projects/{PROJECT}/locations/{REGION}/clusters/{CLUSTER}/instances/{INSTANCE}"
PROJECT_BUCKET = f"gs://{PROJECT}-cymbalgoal-staging"
SERVICE_AGENT = f"service-{PROJECT_NUMBER}@gcp-sa-alloydb.iam.gserviceaccount.com"

# ---- Guarded Cloud Storage -----------------------------------------------------------------
GCS_LOG = []


def _under(uri, prefix):
    p = prefix.rstrip("/")
    return uri == p or uri.startswith(p + "/")


READ_OK = (GCS_READ, GCS_WORK, PROJECT_BUCKET)
WRITE_OK = (RESULTS_PREFIX, PROJECT_BUCKET)


def _guard(args):
    verb = args[0]
    uris = [a for a in args[1:] if a.startswith("gs://")]
    if verb in ("ls", "cat"):
        bad = [u for u in uris if not any(_under(u, p) for p in READ_OK)]
    elif verb == "cp":
        if not any(_under(args[-1], p) for p in WRITE_OK):
            raise PermissionError(f"gcs(): writes only under {WRITE_OK}, refused {args[-1]}")
        bad = [u for u in uris[:-1] if not any(_under(u, p) for p in READ_OK)]
    else:
        raise PermissionError(f"gcs(): '{verb}' refused—only ls, cat and cp are allowed")
    if bad:
        raise PermissionError(f"gcs(): outside the allowed prefixes: {bad}")
    GCS_LOG.append("gcloud storage " + " ".join(args))


def _fx(uri):
    return Path(_FIXTURE) / uri[len("gs://"):]


def gcs(*args):
    _guard(args)
    if _FIXTURE:
        import shutil
        if args[0] == "cat":
            p = _fx(args[-1])
            return subprocess.CompletedProcess(args, 0 if p.exists() else 1,
                                               p.read_text() if p.exists() else "", "" if p.exists() else "No URLs matched")
        if args[0] == "ls":
            p = _fx(args[-1])
            return subprocess.CompletedProcess(args, 0 if p.exists() else 1, args[-1] if p.exists() else "", "")
        d = _fx(args[-1]); d.parent.mkdir(parents=True, exist_ok=True)
        src = args[-2]
        shutil.copy(_fx(src) if src.startswith("gs://") else src, d)
        return subprocess.CompletedProcess(args, 0, "", "")
    return subprocess.run(["gcloud", "storage", *args], capture_output=True, text=True)


def gcs_stream(uri):
    _guard(("cat", uri))
    if _FIXTURE:
        fh = open(_fx(uri), "rb")
        return fh, fh.close
    p = subprocess.Popen(["gcloud", "storage", "cat", uri], stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)

    def _close():
        p.stdout.close(); p.kill(); p.wait()
    return p.stdout, _close


def get_text(uri):
    r = gcs("cat", uri)
    return r.stdout if r.returncode == 0 else None


# ---- Results and handback ------------------------------------------------------------------
HB, RESULTS = {}, {}


def hb(section, line=""):
    HB.setdefault(section, []).append(str(line))


def mins(s):
    if s is None:
        return "—"
    return f"{s:.0f} s" if s < 90 else f"{s / 60:.1f} min"


print(f"project   {PROJECT}  (#{PROJECT_NUMBER})")
print(f"cluster   {CLUSTER}  {REGION}  {SHAPE}")
print(f"instance  {INSTANCE}")
print(f"you       {USER}")
print(f"runtime   {os.cpu_count()} vCPU")
print(f"staging   {PROJECT_BUCKET}" + ("" if gcs('ls', PROJECT_BUCKET).returncode == 0
                                       else "   ⚠️  not found: method D will be skipped"))
hb("2 · Environment", f"project {PROJECT}; cluster {CLUSTER} {REGION}; instance {INSTANCE} {SHAPE}; "
                      f"you {USER}; runtime {os.cpu_count()} vCPU")

---
## 3 · Connect and create the test database

IAM auth through the AlloyDB Python Connector: no password anywhere. Every method below runs
in `cymbalgoal_loadtest`, a database of its own.

In [ ]:
if _FIXTURE:
    import pg8000.dbapi

    def _raw_connect(db):
        return pg8000.dbapi.connect(user="postgres", host=os.environ.get("CG_PGHOST", "localhost"),
                                    port=int(os.environ.get("CG_PGPORT", "5432")), database=db)
else:
    from google.cloud.alloydb.connector import Connector, IPTypes
    connector = Connector()

    def _raw_connect(db):
        return connector.connect(INSTANCE_URI, "pg8000", user=USER, db=db,
                                 enable_iam_auth=True, ip_type=IPTypes.PUBLIC)


def connect(db=DB_NAME):
    c = _raw_connect(db)
    c.autocommit = True
    return c


_tls = threading.local()


def session():
    c = getattr(_tls, "conn", None)
    if c is not None:
        try:
            cur = c.cursor(); cur.execute("SELECT 1"); cur.close()
            return c
        except Exception:
            pass
    _tls.conn = connect()
    return _tls.conn


def _drain(conn, show=False):
    out = []
    for n in list(getattr(conn, "notices", None) or []):
        m = n.get(b"M") if isinstance(n, dict) else None
        if m:
            out.append(m.decode())
    try:
        conn.notices.clear()
    except Exception:
        pass
    if show:
        for m in out:
            print("   ", m)
    return out


def run(sql, params=None, show_notices=False):
    conn = session()
    cur = conn.cursor()
    cur.execute(sql, params) if params else cur.execute(sql)
    cur.close()
    return _drain(conn, show_notices)


def q(sql, params=None):
    conn = session()
    cur = conn.cursor()
    cur.execute(sql, params) if params else cur.execute(sql)
    cols = [d[0] for d in cur.description] if cur.description else []
    rows = cur.fetchall() if cur.description else []
    cur.close()
    _drain(conn)
    return pd.DataFrame(rows, columns=cols)


t0 = time.time()
_c = connect("postgres")
_cur = _c.cursor()
_cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", (DB_NAME,))
if _cur.fetchone() is None:
    _cur.execute(f'CREATE DATABASE "{DB_NAME}"')
    print(f"created database {DB_NAME}")
else:
    print(f"database {DB_NAME} already exists")
_cur.execute("SELECT version()")
print(_cur.fetchone()[0].split(" on ")[0], f"— connected in {time.time() - t0:.1f}s")
_cur.close(); _c.close()

EXTS = ["vector"] if _FIXTURE else ["vector", "alloydb_scann", "google_ml_integration"]
for ext in EXTS:
    run(f"CREATE EXTENSION IF NOT EXISTS {ext}")
if _FIXTURE:   # offline stand-in for the in-database model call: a fixed pseudo-embedding per text
    run("CREATE SCHEMA IF NOT EXISTS ai")
    run('''CREATE OR REPLACE FUNCTION ai.embedding(model text, content text) RETURNS vector
           LANGUAGE sql IMMUTABLE AS $$
             SELECT array_agg(sin(i * (abs(hashtext(content)) % 997 + 1)))::vector
             FROM generate_series(1, 3072) AS i $$''')
display(q("SELECT extname, extversion FROM pg_extension ORDER BY 1"))

---
## 4 · The staged corpus

Reads the manifest and the table DDL that 01 staged, checks the first row of the full file
against the column list, and defines the two loaders every method shares.

In [ ]:
MANIFEST = json.loads(get_text(f"{STAGED_PREFIX}/scout_reports_manifest.json") or "null")
DDL = get_text(f"{STAGED_PREFIX}/scout_reports.sql")
assert MANIFEST and DDL, f"staged corpus not found under {STAGED_PREFIX}/"
COLS = MANIFEST["column_order"]
FILES = MANIFEST["files"]
FULL = "scout_reports.csv.gz"
SEASON_FILES = sorted(k for k in FILES if re.fullmatch(r"scout_reports_\d{4}\.csv\.gz", k))
N_ROWS = FILES[FULL]["rows"]

fh, close = gcs_stream(f"{STAGED_PREFIX}/{FULL}")
with gzip.GzipFile(fileobj=fh) as gz:
    first = next(csv.reader(io.TextIOWrapper(gz, encoding="utf-8")))
close()
assert len(first) == len(COLS), f"first row has {len(first)} fields, manifest lists {len(COLS)}"
DIM = len(first[-1].strip("[]").split(","))

print(f"prompt version {MANIFEST['prompt_version']}, staged {MANIFEST['generated_utc']}")
print(f"full file      {N_ROWS:,} rows, {FILES[FULL]['bytes'] / 1e6:,.1f} MB compressed, {DIM}-dim embeddings")
print(f"season files   {len(SEASON_FILES)}: {SEASON_FILES[0]} … {SEASON_FILES[-1]}")
print(f"columns        {', '.join(COLS)}")
hb("4 · Corpus", f"{MANIFEST['prompt_version']}; {N_ROWS:,} rows; {FILES[FULL]['bytes'] / 1e6:,.1f} MB gz full file; "
                 f"{len(SEASON_FILES)} season files; {DIM} dims")

_REF = re.compile(r"\s+REFERENCES\s+\w+\s*\(\s*\w+\s*\)", re.I)


def make_table(name, with_fk):
    # A fresh, empty copy of scout_reports under another name. Timing a load into a table that
    # already holds rows (or indexes) measures something else.
    ddl = re.sub(r"\bscout_reports\b", name, DDL)
    ddl = "\n".join(l for l in ddl.splitlines() if not l.lstrip().startswith("--"))
    if not with_fk:
        ddl = _REF.sub("", ddl)
    run(f"DROP TABLE IF EXISTS {name} CASCADE")
    run(ddl)


def copy_file(table, uri, cols=COLS, conn=None):
    # Client-side COPY: the file streams from Cloud Storage, is decompressed here, and pg8000
    # pushes it into COPY FROM STDIN. Nothing lands on this runtime's disk.
    own = conn is None
    conn = conn or connect()
    fh, close = gcs_stream(uri)
    try:
        with gzip.GzipFile(fileobj=fh) as gz:
            cur = conn.cursor()
            cur.execute(f"COPY {table} ({', '.join(cols)}) FROM STDIN WITH (FORMAT csv)", stream=gz)
            cur.close()
    finally:
        close()
        if own:
            conn.close()


def count(table):
    return int(q(f"SELECT count(*) AS n FROM {table}")["n"][0])


def verify(table):
    n = count(table)
    d = int(q(f"SELECT vector_dims(report_embedding) AS d FROM {table} LIMIT 1")["d"][0]) if n else 0
    return n, d

---
## 5 · Base tables

The eight CymbalGoal tables, loaded the way the earlier CymbalGoal lab loads them: `schema.sql`,
then a client-side `COPY` per table. Every module needs them, so this time is part of any
module's total. They also give `scout_reports` its foreign keys to `players` and `clubs`.

Player and club profiles (and their embeddings) are not loaded: the scouting reports are this
workshop's vector corpus.

In [ ]:
S = "5 · Base tables"
BASE_TABLES = ["competitions", "clubs", "players", "games",
               "appearances", "game_events", "player_valuations", "transfers"]
BASE_OK = False
RESULTS["base"] = None


def _base_present():
    have = set(q("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public'").table_name)
    return [t for t in BASE_TABLES if t in have]


if not RUN_BASE:
    print("skipped—RUN_BASE is False")
    hb(S, "skipped")
else:
    present = _base_present()
    if present and FRESH_BASE:
        for t in reversed(BASE_TABLES):
            run(f"DROP TABLE IF EXISTS {t} CASCADE")
        present = []
    if len(present) == len(BASE_TABLES) and all(count(t) for t in BASE_TABLES):
        BASE_OK = True
        print("base tables already loaded—not timed this run (FRESH_BASE = True to reload and time)")
        hb(S, "already loaded, not timed (FRESH_BASE = True to time)")
    else:
        if present:
            raise SystemExit(f"{len(present)} of {len(BASE_TABLES)} base tables exist—a half-built schema. "
                             "Set FRESH_BASE = True and rerun.")
        bman = json.loads(get_text(f"{GCS_READ}/manifest.json"))["staged_files"]
        bcols = {t: [c for c in bman[t]["column_order"] if c not in ("profile_text", "profile_embedding")]
                 for t in BASE_TABLES}
        t0 = time.time()
        run(get_text(f"{GCS_READ}/schema.sql"))
        per = {}
        for t in BASE_TABLES:
            t1 = time.time()
            copy_file(t, f"{GCS_READ}/{t}.csv.gz", bcols[t])
            per[t] = (count(t), time.time() - t1)
            print(f"  {t:<20}{per[t][0]:>10,} rows  {per[t][1]:>6.1f}s")
        RESULTS["base"] = time.time() - t0
        BASE_OK = True
        print(f"\nbase tables loaded in {mins(RESULTS['base'])}")
        hb(S, f"{mins(RESULTS['base'])}: " + "; ".join(f"{t} {n:,} {s:.0f}s" for t, (n, s) in per.items()))

---
## 6 · Method A—COPY, one file

The full corpus, one stream, through this runtime. The simplest thing that works, and the
baseline for everything below.

In [ ]:
METHODS = {}


def record(key, label, seconds, table, note=""):
    n, d = verify(table) if seconds is not None else (0, 0)
    ok = seconds is not None and n == N_ROWS
    METHODS[key] = {"label": label, "seconds": seconds, "rows": n, "dims": d, "ok": ok, "table": table, "note": note}
    mbps = FILES[FULL]["bytes"] / 1e6 / seconds if seconds else 0
    line = (f"{label}: {mins(seconds)}—{n:,} rows, {d} dims, {mbps:,.1f} MB/s compressed"
            + (f"; {note}" if note else "") + ("" if ok else "  🔴 NOT a complete load"))
    print(("✅ " if ok else "🔴 ") + line)
    hb("Load methods", line)


S = "6 · A"
if not RUN_COPY_ONE:
    print("skipped—RUN_COPY_ONE is False")
else:
    make_table("sr_copy_one", BASE_OK)
    t0 = time.time()
    copy_file("sr_copy_one", f"{STAGED_PREFIX}/{FULL}")
    record("A", "A · COPY, one file", time.time() - t0, "sr_copy_one")

---
## 7 · Method B—COPY, per-season files in parallel

The same bytes in 14 files, `PARALLEL_WORKERS` streams at a time, each on its own connection.
If the single stream in A is limited by one connection rather than by the database, this is
where it shows.

In [ ]:
S = "7 · B"
if not RUN_COPY_PARALLEL:
    print("skipped—RUN_COPY_PARALLEL is False")
else:
    make_table("sr_copy_par", BASE_OK)
    errors = []

    def _one(name):
        t1 = time.time()
        try:
            copy_file("sr_copy_par", f"{STAGED_PREFIX}/{name}")
            return name, time.time() - t1, ""
        except Exception as e:  # noqa: BLE001
            return name, time.time() - t1, f"{type(e).__name__}: {str(e)[:200]}"

    t0 = time.time()
    with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as pool:
        parts = list(pool.map(_one, SEASON_FILES))
    el = time.time() - t0
    errors = [f"{n}: {e}" for n, _, e in parts if e]
    slowest = max(parts, key=lambda x: x[1])
    record("B", f"B · COPY, {len(SEASON_FILES)} files × {PARALLEL_WORKERS} workers", el, "sr_copy_par",
           f"slowest file {slowest[0]} {slowest[1]:.0f}s" + (f"; errors {errors[:2]}" if errors else ""))

---
## 8 · AlloyDB import—the shared machinery

`gcloud alloydb clusters import` is a **server-side** load: AlloyDB reads the file from Cloud
Storage itself, so nothing passes through this runtime. This calls the same API directly, so the
notebook can poll the operation and time it.

Two things decide whether it works:

- **The AlloyDB service agent reads the file, not you.** It needs `storage.objectViewer` on the
  source bucket.
- **The database user.** The import runs `COPY` as the `user` you name. It tries your IAM user
  first (you own the table), then `postgres`.

In [ ]:
if not _FIXTURE:
    import google.auth
    from google.auth.transport.requests import AuthorizedSession
    _creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    API = AuthorizedSession(_creds)
CLUSTER_URL = f"https://alloydb.googleapis.com/v1/projects/{PROJECT}/locations/{REGION}/clusters/{CLUSTER}"


def _fixture_import(uri, table):
    # Offline stand-in for the server-side import (the offline test server can't read the file).
    copy_file(table, uri)
    return {"ok": True, "user": USER, "server_s": None, "errors": []}


def alloydb_import(uri, table, timeout_s=3600):
    if _FIXTURE:
        return _fixture_import(uri, table)
    errors = []
    for user in (USER, "postgres"):
        if user == "postgres":
            try:
                run(f"GRANT ALL ON TABLE {table} TO postgres")
            except Exception as e:  # noqa: BLE001
                errors.append(f"GRANT to postgres: {str(e)[:150]}")
        body = {"gcsUri": uri, "database": DB_NAME, "user": user,
                "csvImportOptions": {"table": table, "columns": COLS}}
        r = API.post(f"{CLUSTER_URL}:import", json=body)
        if r.status_code != 200:
            errors.append(f"{user}: HTTP {r.status_code} {r.text[:300]}")
            continue
        op_name = r.json()["name"]
        t_end = time.time() + timeout_s
        while time.time() < t_end:
            op = API.get(f"https://alloydb.googleapis.com/v1/{op_name}").json()
            if op.get("done"):
                break
            time.sleep(5)
        else:
            errors.append(f"{user}: still running after {timeout_s}s ({op_name})")
            return {"ok": False, "user": user, "server_s": None, "errors": errors}
        if "error" in op:
            errors.append(f"{user}: {json.dumps(op['error'])[:400]}")
            if count(table):          # a partial load would poison the next attempt
                run(f"TRUNCATE {table}")
            continue
        md_ = op.get("metadata", {})
        srv = None
        if md_.get("createTime") and md_.get("endTime"):
            srv = (pd.Timestamp(md_["endTime"]) - pd.Timestamp(md_["createTime"])).total_seconds()
        return {"ok": True, "user": user, "server_s": srv, "errors": errors}
    return {"ok": False, "user": None, "server_s": None, "errors": errors}


GRANT_CLASS_DEMO = (f"gcloud storage buckets add-iam-policy-binding gs://class-demo "
                    f"--member=serviceAccount:{SERVICE_AGENT} --role=roles/storage.objectViewer")
print("import machinery ready")
print(f"AlloyDB service agent: {SERVICE_AGENT}")

---
## 9 · Method C—import straight from the course bucket

Works only if this project's AlloyDB service agent can read `class-demo`. In development you can
grant that (the command is printed if it fails). **At an event it would mean granting every
student project's service agent, which a student project cannot do for itself**, so a success
here mainly tells us how fast the import is when it reads from the course bucket.

In [ ]:
S = "9 · C"
if not RUN_IMPORT_DIRECT:
    print("skipped—RUN_IMPORT_DIRECT is False")
else:
    make_table("sr_import_direct", BASE_OK)
    t0 = time.time()
    res = alloydb_import(f"{STAGED_PREFIX}/{FULL}", "sr_import_direct")
    el = time.time() - t0
    if res["ok"]:
        record("C", "C · Import from class-demo", el, "sr_import_direct",
               f"as {res['user']}" + (f"; server-side {res['server_s']:.0f}s" if res["server_s"] else ""))
    else:
        METHODS["C"] = {"label": "C · Import from class-demo", "seconds": None, "ok": False, "errors": res["errors"]}
        print("🔴 import from class-demo failed:")
        for e in res["errors"]:
            print("   ", e)
        if any("403" in e or "ermission" in e or "storage" in e.lower() for e in res["errors"]):
            print(f"\nTo let this project's service agent read the course bucket (bucket owner, Cloud Shell):\n  {GRANT_CLASS_DEMO}")
        hb("Load methods", f"C · Import from class-demo: FAILED—{' | '.join(res['errors'])[:600]}")

---
## 10 · Method D—copy to the project bucket, then import

The route that works at an event without touching the course bucket's permissions: Terraform
makes an in-project staging bucket and lets the AlloyDB service agent read it. Copying
bucket-to-bucket is server-side, so it is timed separately from the import.

In [ ]:
S = "10 · D"
if not RUN_IMPORT_PROJECT:
    print("skipped—RUN_IMPORT_PROJECT is False")
elif gcs("ls", PROJECT_BUCKET).returncode != 0:
    print(f"skipped—{PROJECT_BUCKET} does not exist (apply the Terraform)")
    hb("Load methods", f"D · Import via project bucket: skipped, {PROJECT_BUCKET} missing")
else:
    t0 = time.time()
    r = gcs("cp", f"{STAGED_PREFIX}/{FULL}", f"{PROJECT_BUCKET}/{FULL}")
    t_copy = time.time() - t0
    if r.returncode != 0:
        print(f"🔴 copy to {PROJECT_BUCKET} failed: {r.stderr[:400]}")
        hb("Load methods", f"D · copy to project bucket FAILED: {r.stderr[:300]}")
    else:
        print(f"copied to {PROJECT_BUCKET} in {t_copy:.0f}s")
        make_table("sr_import_project", BASE_OK)
        t1 = time.time()
        res = alloydb_import(f"{PROJECT_BUCKET}/{FULL}", "sr_import_project")
        t_imp = time.time() - t1
        if res["ok"]:
            record("D", "D · Copy + import via project bucket", t_copy + t_imp, "sr_import_project",
                   f"copy {t_copy:.0f}s + import {t_imp:.0f}s as {res['user']}"
                   + (f" (server-side {res['server_s']:.0f}s)" if res["server_s"] else ""))
        else:
            METHODS["D"] = {"label": "D · Copy + import via project bucket", "seconds": None, "ok": False,
                            "errors": res["errors"]}
            print("🔴 import from the project bucket failed:")
            for e in res["errors"]:
                print("   ", e)
            hb("Load methods", f"D · Copy + import: FAILED—{' | '.join(res['errors'])[:600]}")

---
## 11 · Indexes

Built **after** the load, never before: a ScaNN index cannot be built on an empty table, and an
index that exists during a bulk load makes every row an index insert.

- **B-tree** indexes on `player_id` and `season`, plus `ANALYZE`, on the full table.
- **ScaNN** on three sizes, each its own table copied from a full load (the copy isn't timed):
  `num_leaves = √rows` and `quantizer = 'sq8'`, as `scout_reports.sql` recommends.

In [ ]:
S = "11 · Indexes"
INDEX = {}
SRC = next((m["table"] for k, m in sorted(METHODS.items()) if m.get("ok")), None)
if not RUN_INDEXES:
    print("skipped—RUN_INDEXES is False")
elif SRC is None:
    print("skipped—no method produced a complete table")
    hb(S, "skipped: no complete table")
else:
    print(f"source table: {SRC}\n")
    t0 = time.time()
    run(f"CREATE INDEX ON {SRC} (player_id)")
    run(f"CREATE INDEX ON {SRC} (season)")
    run(f"ANALYZE {SRC}")
    INDEX["btree"] = time.time() - t0
    print(f"B-tree (player_id, season) + ANALYZE: {INDEX['btree']:.1f}s")
    hb(S, f"B-tree player_id + season + ANALYZE on {SRC}: {INDEX['btree']:.1f}s")

    seasons = sorted(int(x) for x in q(f"SELECT DISTINCT season FROM {SRC}").season)
    run(f"SET maintenance_work_mem = '{MAINTENANCE_WORK_MEM}'")
    for label, k in SCANN_SIZES:
        tbl = "sr_scann_" + ("all" if k is None else str(k))
        where = "" if k is None else f"WHERE season >= {seasons[-k]}"
        run(f"DROP TABLE IF EXISTS {tbl}")
        run(f"CREATE TABLE {tbl} AS SELECT * FROM {SRC} {where}")
        n = count(tbl)
        leaves = max(1, int(math.sqrt(n)))
        if _FIXTURE:   # pgvector offline can't index 3072 dims; a B-tree keeps the flow honest
            ddl = f"CREATE INDEX {tbl}_scann ON {tbl} (report_id)"
        else:
            ddl = (f"CREATE INDEX {tbl}_scann ON {tbl} USING scann (report_embedding cosine) "
                   f"WITH (num_leaves = {leaves}, quantizer = 'sq8')")
        t1 = time.time()
        notes = run(ddl, show_notices=False)
        INDEX[label] = {"rows": n, "leaves": leaves, "seconds": time.time() - t1, "table": tbl}
        print(f"ScaNN {label:<12}{n:>8,} rows  {leaves:>4} leaves  {INDEX[label]['seconds']:>6.1f}s")
        for m in notes[:3]:
            print("     ", m)
        hb(S, f"ScaNN {label}: {n:,} rows, {leaves} leaves, {INDEX[label]['seconds']:.1f}s"
              + (f"—{notes[0][:160]}" if notes else ""))
    run("RESET maintenance_work_mem")

---
## 12 · Search check

Proof the corpus works end to end: each query is embedded in the database with `ai.embedding()`,
then searched against the full ScaNN-indexed table. Recall@10 compares ScaNN's top ten with an
exact scan of every row.

In [ ]:
S = "12 · Search"
TOP = {}
tbl = INDEX.get("all seasons", {}).get("table")
if not RUN_SEARCH:
    print("skipped—RUN_SEARCH is False")
elif not tbl:
    print("skipped—needs the all-seasons ScaNN table from section 11")
    hb(S, "skipped")
else:
    plan = "\n".join(q(f"EXPLAIN SELECT report_id FROM {tbl} ORDER BY report_embedding <=> "
                       f"ai.embedding('{EMBED_MODEL}', 'test')::vector LIMIT 10").iloc[:, 0])
    uses = "scann" in plan.lower() or _FIXTURE
    print(("✅ the plan uses the ScaNN index" if uses else "🔴 the plan does NOT use the ScaNN index") + "\n")
    hb(S, f"plan uses ScaNN: {uses}")
    for text in SEARCH_QUERIES:
        t0 = time.time()
        vec = q("SELECT ai.embedding(%s::text, %s::text)::vector::text AS v", (EMBED_MODEL, text))["v"][0]
        t_emb = time.time() - t0
        sql = (f"SELECT report_id, recommendation, headline FROM {tbl} "
               f"ORDER BY report_embedding <=> '{vec}'::vector LIMIT 10")
        t1 = time.time()
        ann = q(sql)
        t_ann = time.time() - t1
        run("SET enable_indexscan = off")
        exact = q(sql)
        run("RESET enable_indexscan")
        recall = len(set(ann.report_id) & set(exact.report_id)) / 10
        print(f"“{text}”\n  embed {t_emb * 1000:.0f} ms, search {t_ann * 1000:.0f} ms, recall@10 {recall:.0%}")
        for _, r in ann.head(3).iterrows():
            print(f"    [{r.recommendation}] {r.headline}  ({r.report_id})")
        print()
        hb(S, f"“{text}”: embed {t_emb * 1000:.0f} ms, search {t_ann * 1000:.0f} ms, recall@10 {recall:.0%}; top: "
              + " | ".join(f"[{r.recommendation}] {r.headline}" for _, r in ann.head(3).iterrows()))

---
## 13 · Against the budget, and handback

What a module's load would cost a student, per method: base tables + scouting reports + indexes
(B-tree and the all-seasons ScaNN build), against the **11-minute** budget left after cluster
creation. The results are also saved as JSON under `load-tests/` in the course bucket.

One box below holds everything Claude needs, with a **Copy** button.

In [ ]:
import html as _html
from IPython.display import HTML, display

base_s = RESULTS.get("base")
idx_s = (INDEX.get("btree") or 0) + (INDEX.get("all seasons", {}).get("seconds") or 0)
rows = []
for k in sorted(METHODS):
    m = METHODS[k]
    if not m.get("ok"):
        rows.append((m["label"], None, "failed or incomplete"))
        continue
    total = (base_s or 0) + m["seconds"] + idx_s
    rows.append((m["label"], total, "fits" if total / 60 <= BUDGET_MIN else "OVER"))
print(f"budget {BUDGET_MIN:.0f} min; base {mins(base_s)}"
      + ("" if base_s is not None else " (not timed this run)") + f"; indexes {mins(idx_s)}\n")
for label, total, verdict in rows:
    print(f"  {label:<44}{mins(total):>10}   {verdict}")
for label, total, verdict in rows:
    hb("Budget", f"{label}: total {mins(total)}—{verdict}")
hb("Budget", f"budget {BUDGET_MIN:.0f} min; base {mins(base_s)}; indexes {mins(idx_s)}")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = {"run_utc": stamp, "project": PROJECT, "shape": SHAPE, "runtime_vcpu": os.cpu_count(),
       "corpus": {"rows": N_ROWS, "bytes_gz": FILES[FULL]["bytes"], "dims": DIM,
                  "prompt_version": MANIFEST["prompt_version"]},
       "base_s": base_s, "methods": METHODS, "indexes": INDEX, "budget_min": BUDGET_MIN}
p = BASE / f"load-test-{stamp}.json"
p.write_text(json.dumps(OUT, indent=2, default=str))
r = gcs("cp", str(p), f"{RESULTS_PREFIX}/{p.name}")
saved = r.returncode == 0
print(f"\nresults {'saved to ' + RESULTS_PREFIX + '/' + p.name if saved else 'NOT saved to the bucket: ' + r.stderr[:200]}")

lines = ["=" * 70, "CYMBALGOAL LOAD TEST—02 HANDBACK", f"run {stamp}", "=" * 70]
for sec, ls_ in HB.items():
    lines += ["", f"## {sec}", *ls_]
writes = [c for c in GCS_LOG if c.split()[2] == "cp"]
bad = [c for c in writes if not any(_under(c.split()[-1], w) for w in WRITE_OK)]
lines += ["", "## Bucket audit",
          f"calls {len(GCS_LOG)} ({dict(Counter(c.split()[2] for c in GCS_LOG))}); writes {len(writes)}, "
          f"all under the results prefix or the project bucket: {not bad}; results saved: {saved}", "=" * 70]
TEXT = "\n".join(lines)
(BASE / "handback.txt").write_text(TEXT)
assert not bad, f"a write left the allowed prefixes: {bad}"

display(HTML(f'''
<div style="font-family:system-ui,sans-serif;max-width:100%">
  <button id="cg-copy" style="font-size:15px;font-weight:700;padding:10px 20px;border-radius:8px;border:0;
      background:#1f8a6d;color:#fff;cursor:pointer">Copy handback</button>
  <span id="cg-msg" style="margin-left:12px;font-weight:600;color:#1f8a6d"></span>
  <textarea id="cg-box" readonly style="display:block;width:100%;height:340px;margin-top:10px;
      font:12px/1.4 ui-monospace,Menlo,monospace;padding:10px;box-sizing:border-box">{_html.escape(TEXT)}</textarea>
</div>
<script>
(function() {{
  const b = document.getElementById("cg-copy"), t = document.getElementById("cg-box"), m = document.getElementById("cg-msg");
  b.addEventListener("click", async () => {{
    try {{ await navigator.clipboard.writeText(t.value); m.textContent = "Copied—paste it into the chat."; }}
    catch (e) {{ t.focus(); t.select(); m.textContent = "Clipboard blocked: the text is selected, use Edit → Copy."; }}
  }});
}})();
</script>'''))
print(f"{len(TEXT):,} characters; also saved to {BASE / 'handback.txt'}")